In [2]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Get_missing_terminal_cappings)=
# Get missing terminal cappings

*Identifying missing terminal cappings in peptides and proteins.*

The function {func}`molsysmt.build.get_missing_terminal_cappings` checks whether the N- or C-termini of peptide and protein chains lack terminal capping groups (such as `ACE` or `NME`) or required terminal atoms (such as `OXT`).

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.build.get_missing_terminal_cappings`.
:::

**Recognize chemical ends from stored bonds.** The experimental {func}`molsysmt.chemical_states.get_peptide_termini` follows named N-CA-C motifs and inter-group C-N links, independently of atom/group storage order or group ID labels. Complete chain context precedes endpoint filtering. Connected cycles have no free ends; disconnected, branched or unsupported backbones are refused. Missing-OXT assessment and terminal addition reuse this rule, while placement still does not assign complete chemistry. See {ref}`Getting peptide termini <Tutorial_Get_peptide_termini>`.

PDBFixer terminal assessment also retains the complete source before filtering selected groups. Numeric inputs select source group indices; query strings use the supplied syntax. An internal selection never becomes an artificial chain end.


## Basic usage

Let's show how this function works using a protein structure loaded from PDB entry `1BRS`:


**Choosing a source population.**

`add_missing_terminal_cappings` interprets numeric selections as source **atom
indices**; a Boolean mask spans the source atom axis. Resolve a query with
`msm.select(molsys, selection=...)` to obtain the equivalent indices. The input
syntax changes query parsing, not the meaning of an integer list. Empty
selections return independent unchanged copies.

Native local heavy-atom/OXT/N-bound-H completion enables whole groups touched by
the source atoms. ACE/NME requests enable complete peptide/protein covalent
components touched by those atoms. Atom insertion can reorder atoms, so the
composition transports group membership and re-evaluates component populations;
it does not reuse stale atom indices or equate components with stable molecules.
An internal selected group remains internal. Existing state, placement,
attribute-loss and interaction-invalidation limits still apply.

`get_missing_terminal_cappings` is a **group** assessment: its integer selections
are group indices. It identifies termini from complete chains and then filters
the report; a selection edge is not a chemical terminus.


In [3]:
import molsysmt as msm


In [4]:
molsys = msm.convert(
    msm.systems["Barnase-Barstar"]["1brs.bcif.gz"],
    selection='molecule_type=="protein"')


Now we run {func}`molsysmt.build.get_missing_terminal_cappings` to identify missing terminal groups:


In [5]:
missing_caps = msm.build.get_missing_terminal_cappings(molsys)


In [6]:
missing_caps


{}

:::{warning}
The native recognizer leaves disconnected Barstar chains unassessed. An empty
missing-OXT dictionary therefore does not establish complete terminal chemistry.
The original label-based expectation at group 412 is not a stored-backbone
reference. Inspect the explicit coverage report before choosing any repair.
:::

The general {func}`molsysmt.chemical_states.get_peptide_termini` reports the
unassessed chain positions without inventing fragment termini:


In [7]:
termini = msm.chemical_states.get_peptide_termini(
    molsys, on_ambiguous="report")
termini["unassessed_chain_indices"].tolist()

[3, 4]

We can select terminal oxygen atoms (`OXT`) using {func}`molsysmt.basic.select`:


In [8]:
msm.select(molsys, selection='atom_name=="OXT"')


[863, 1741, 2580, 3938, 4637]

:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Convert <Tutorial_Convert>`: Convert molecular systems between different forms with {func}`molsysmt.basic.convert`.
- {ref}`Get missing terminal cappings <Tutorial_Get_missing_terminal_cappings>`: Identify missing terminal capping groups with {func}`molsysmt.build.get_missing_terminal_cappings`.
- {ref}`Select <Tutorial_Select>`: Select atom indices matching specified query syntax with {func}`molsysmt.basic.select`.
- {ref}`Add missing terminal cappings <Tutorial_Add_missing_terminal_cappings>`: Add terminal cappings to peptides or proteins with {func}`molsysmt.build.add_missing_terminal_cappings`.
:::
